# WEFESiteAnalyst Template
## implemented as Jupyter Notebook

This Jupyter Notebook collects, analyzes, and visualizes environmental and socioeconomic data for the planning of integrated water, energy, food, and environment systems from open servers. This file is the template jupyter notebbook for applying the WEFESiteAnalyst. Kindly put the notebook file in a new folder for your case study when you start collecting data for a new location.

In [ ]:
# imports
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import os
from math import sqrt
import numpy as np
import folium
#import geojson
import overpy
import geemap
import geemap.foliumap as gfolium
import json
import requests
import ee
from geemap import geojson_to_ee, ee_to_geojson
from ipyleaflet import GeoJSON
from geojson import Point, Feature, FeatureCollection, dump

from geopandas import GeoDataFrame

import shapely
from shapely.geometry import Polygon, shape
from shapely.geometry import LineString

# Save the current working directory
original_dir = os.getcwd()

# import files from source
os.chdir("../../src/")
import era5
import ee_layer

# Return to the original directory
os.chdir(original_dir)

In [ ]:
# set start and end date (end date will be included
# in the time period for which data is downloaded)
start_date, end_date = '2022-01-01', '2022-12-31'  # time in UTC choose start date one day before time of interest
# for position east of 0° meridian for covering all hours of interest

In [ ]:
# initiate earth engine


# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize()

In [ ]:
# provide case study name and coordiantes

name = input("Enter the name of the case study: ")
lat = float(input("Enter the latitude: "))
lon = float(input("Enter the longitude: "))

center_ee = ee.Geometry.Point(lon, lat)


m = gfolium.Map(
    center=[lat, lon], zoom=14, toolbar_control=False, layers_control=True
)

m.add_basemap('SATELLITE')

m.addLayer(center_ee, {'color': 'cyan', 'weight': 1, 'opacity': 0.8}, "center")
m.addLayer(dws1, {'color':'blue', 'weight':1, 'opacity':0.8}, "drinking water source")
m.addLayer(ee_aoi,{'color': 'red', 'weight': 1, 'opacity': 0.3, 'fillColor' : 'red',  'fillOpacity' : 0}, "settlement")

m


In [ ]:
## TERRAIN MAP
# Define the location for which you want to create the elevation map (given abo)

center = ee.Geometry.Point(lon, lat)
buffer = center.buffer(5000)

# create GeoJSON for the specific point
center = {
  "type": "Point",
  "coordinates": [lon, lat]
}

# Get the elevation at the map center
elevation = ee.Image('USGS/SRTMGL1_003').sample(center)

# Get the elevation value at the map center
elevation_value = elevation.get('elevation').getInfo()

# Print the elevation value
print(f'Elevation at {lon}, {lat}: {elevation_value} meters')


# Create a geemap map centered on the location
#terrain_map = geemap.Map(center=[lat, lon], zoom=16)
terrain_map = gfolium.Map(center=[lat, lon], zoom=16)
#terrain_map.add_layer(center)

# Use the 'SRTM 30m Digital Elevation Database' dataset to get elevation dataset

elevation = ee.Image('USGS/SRTMGL1_003').clip(buffer);
slope = ee.Terrain.slope(elevation);

# Calculate aspect. Units are degrees where 0=N, 90=E, 180=S, 270=W.
aspect = ee.Terrain.aspect(elevation);
terrain = ee.Terrain.products(elevation);

vis_params = {
    'min': 0,
    'max': 200,
    'palette': ['006633', 'E5FFCC', '662A00', 'D8D8D8', 'F5F5F5'],
}

vis_params_slope = {
    'min': 0,
    'max': 20,
    'palette': ['FFFFFF', '000000'],
}

terrain_map.addLayer(elevation, vis_params, 'Elevation')
terrain_map.addLayer(slope,{'min': 0, 'max': 20, 'palette': ['FFFFFF', '000000']}, 'Slope')
terrain_map.addLayer(aspect,{'min': 0, 'max': 360, 'palette': ['006633', 'E5FFCC', '662A00', 'D8D8D8', 'F5F5F5']}, 'Aspect')
terrain_map.addLayer(terrain.select('hillshade'), {min: 0, max: 255}, 'Hillshade')
terrain_map.addLayer(ee_aoi,{'color': 'red', 'weight': 1, 'opacity': 0.3, 'fillColor' : 'red',  'fillOpacity' : 0}, "settlement")
terrain_map.addLayer(center_ee,{'color': 'cyan', 'weight': 1, 'opacity': 0.7}, "center")
terrain_map.addLayer(dws1, {'color':'blue', 'weight':1, 'opacity':0.8}, "drinking water source")


terrain_map.add_basemap('SATELLITE')

# Make pixels with elevation below sea level transparent -> please implement
# elv_img = srtm.updateMask(srtm.gt(0))

colors = vis_params['palette']
vmin = vis_params['min']
vmax = vis_params['max']

terrain_map.add_colorbar_branca(vis_params = vis_params, colors=colors, vmin=vmin, vmax=vmax, layer_name="Elevation")
#terrain_map.add_colorbar_branca(vis_params = vis_params_slope, colors = vis_params_slope['palette'], vmin= vis_params_slope['min'], vmax= vis_params_slope['max'], layer_name="slope")

# Display the map
terrain_map


In [ ]:
## LANDUSE MAP
# Define the location for which you want to create the map

center = ee.Geometry.Point(lon, lat)
buffer = center.buffer(5000)

landuse_map = gfolium.Map(center=[lat, lon], zoom=14)
dataset = ee.ImageCollection("ESA/WorldCover/v100").first()
landuse_map.addLayer(dataset, {'bands': ['Map']}, 'ESA Land Cover')
landuse_map.add_legend(builtin_legend='ESA_WorldCover')
landuse_map.add_basemap('SATELLITE')
landuse_map.addLayer(ee_aoi,{'color': 'red', 'weight': 1, 'opacity': 0.3, 'fillColor' : 'red',  'fillOpacity' : 0}, "settlement")
landuse_map.addLayer(center_ee,{'color': 'cyan', 'weight': 1, 'opacity': 1}, "center")
landuse_map.addLayer(dws1, {'color':'blue', 'weight':1, 'opacity':0.8}, "drinking water source")
landuse_map

In [ ]:
import folium
import overpy

dws1 = ee.Geometry.Point(lon_dw, lat_dw)
hm.addLayer(dws1, {'color':'blue', 'weight':1, 'opacity':0.7}, "drinking water source")

bbox_size = 0.01

# Calculate the bounds of the bounding box
bounds = [lat-bbox_size, lon-bbox_size, lat+bbox_size, lon+bbox_size]

# Connect to the Overpass API
api = overpy.Overpass()

# Query for all water areas including rivers, creeks, reservoirs, etc. within the bounding box
query = f"""
(
  way["waterway"~"river|stream|canal|ditch|drain|brook|creek"]({bounds[0]},{bounds[1]},{bounds[2]},{bounds[3]});
  way["natural"="water"]({bounds[0]},{bounds[1]},{bounds[2]},{bounds[3]});
  way["landuse"="reservoir"]({bounds[0]},{bounds[1]},{bounds[2]},{bounds[3]});
  way["natural"="bay"]({bounds[0]},{bounds[1]},{bounds[2]},{bounds[3]});
  way["landuse"="basin"]({bounds[0]},{bounds[1]},{bounds[2]},{bounds[3]});
);
(._;>;);
out body;
"""

# Execute the query
result = api.query(query)

# Create a Folium map centered at the center of the bounding box
m = gfolium.Map(location=[lat, lon], zoom_start=14, Tiles=None)

# Add the water bodies to the map
for way in result.ways:
    folium.PolyLine(locations=[(node.lat, node.lon) for node in way.nodes], color="blue", weight=2.5, opacity=1).add_to(m)

# Display the map
m


In [ ]:
# Hydrology Map

center = ee.Geometry.Point(lon, lat)


dws1 = ee.Geometry.Point(lon_dw, lat_dw)

hm = gfolium.Map(
    center=[lat, lon], zoom=17, toolbar_control=False, layers_control=True
)

hm.add_basemap('SATELLITE')

hm.addLayer(center_ee, {'color': 'cyan', 'weight': 1, 'opacity': 0.7}, "center")
hm.addLayer(dws1, {'color':'blue', 'weight':1, 'opacity':0.7}, "drinking water source")


# "amenity":"drinking_water"
#waterway=river waterway=streamwaterway=tidal_channel waterway=canal waterway=ditch waterway=drain waterway=pressurised
#natural=water
#water=*
hm

In [ ]:
## Soil Properties

#soil organic carbon
soc_mean = ee.Image("projects/soilgrids-isric/soc_mean")
print(type(soc_mean))

# Soil salinity
# Soil depth
# soil bulk density
bdod_mean = ee.Image("projects/soilgrids-isric/bdod_mean")
# Soil Cation Exchange Capacity
cec_mean = ee.Image("projects/soilgrids-isric/cec_mean")
# Soil moisture
#nitrogen
nitrogen_mean = ee.Image("projects/soilgrids-isric/nitrogen_mean")
# Soil pH (in H2O)
phh2o_mean = ee.Image("projects/soilgrids-isric/phh2o_mean")
# soil texture
clay_mean = ee.Image("projects/soilgrids-isric/clay_mean")
silt_mean = ee.Image("projects/soilgrids-isric/silt_mean")
sand_mean = ee.Image("projects/soilgrids-isric/sand_mean")

location = {
  "type": "Point",
  "coordinates": [lon, lat]
}

# Extract the soil properties for the location
soc_mean = soc_mean.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=ee.Geometry.Point(location["coordinates"]),
    scale=250
).getInfo()

bdod_mean = bdod_mean.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=ee.Geometry.Point(location["coordinates"]),
    scale=250
).getInfo()

cec_mean = cec_mean.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=ee.Geometry.Point(location["coordinates"]),
    scale=250
).getInfo()

nitrogen_mean = nitrogen_mean.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=ee.Geometry.Point(location["coordinates"]),
    scale=250
).getInfo()

phh2o_mean = phh2o_mean.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=ee.Geometry.Point(location["coordinates"]),
    scale=250
).getInfo()

clay_mean = clay_mean.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=ee.Geometry.Point(location["coordinates"]),
    scale=250
).getInfo()

silt_mean = silt_mean.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=ee.Geometry.Point(location["coordinates"]),
    scale=250
).getInfo()
index_list = ['0-5cm', '100-200cm', '15-30cm', '30-60cm', '5-15cm', '60-100cm']

sand_mean = sand_mean.reduceRegion(
    reducer=ee.Reducer.first(),
    geometry=ee.Geometry.Point(location["coordinates"]),
    scale=250
).getInfo()

# Convert the soil properties to a pandas dataframe
soc_mean_df = pd.DataFrame.from_dict(soc_mean, orient='index', columns=["soil organic carbon [dg/kg]"])
soc_mean_df['soil_depth'] = index_list
soc_mean_df = soc_mean_df.set_index(keys='soil_depth')

bdod_mean_df = pd.DataFrame.from_dict(bdod_mean, orient='index', columns=["bulk density [cg/cm³]"])
bdod_mean_df['soil_depth'] = index_list
bdod_mean_df = bdod_mean_df.set_index(keys='soil_depth')

cec_mean_df  = pd.DataFrame.from_dict(cec_mean, orient='index', columns=["cation exchange capacity at pH 7 [mmol(c)/kg]"])
cec_mean_df['soil_depth'] = index_list
cec_mean_df = cec_mean_df.set_index(keys='soil_depth')

nitrogen_mean_df = pd.DataFrame.from_dict(nitrogen_mean, orient='index', columns=["nitrogen content [mg/kg]"])
nitrogen_mean_df['soil_depth'] = index_list
nitrogen_mean_df = nitrogen_mean_df.set_index(keys='soil_depth')

phh2o_mean_df = pd.DataFrame.from_dict(phh2o_mean, orient='index', columns=["pH of soil water [pH*10]"])
phh2o_mean_df['soil_depth'] = index_list
phh2o_mean_df = phh2o_mean_df.set_index(keys='soil_depth')

clay_mean_df = pd.DataFrame.from_dict(clay_mean, orient='index', columns=["clay content [g/kg]"])
clay_mean_df['soil_depth'] = index_list
clay_mean_df = clay_mean_df.set_index(keys='soil_depth')

silt_mean_df = pd.DataFrame.from_dict(silt_mean, orient='index', columns=["silt content [g/kg]"])
silt_mean_df['soil_depth'] = index_list
silt_mean_df = silt_mean_df.set_index(keys='soil_depth')

sand_mean_df = pd.DataFrame.from_dict(clay_mean, orient='index', columns=["sand content [g/kg]"])
sand_mean_df['soil_depth'] = index_list
sand_mean_df = sand_mean_df.set_index(keys='soil_depth')

soil_properties_df = pd.concat((soc_mean_df, bdod_mean_df, cec_mean_df, nitrogen_mean_df, phh2o_mean_df, clay_mean_df, silt_mean_df, sand_mean_df), axis=1)
print(soil_properties_df)


def soil_texture(row):
    clay =row["clay content [g/kg]"]
    silt =row["silt content [g/kg]"]
    sand =row["sand content [g/kg]"]
    
    total = clay + silt + sand
    clay_percent = clay / total * 100
    silt_percent = silt / total * 100
    sand_percent = sand / total * 100

    if sand_percent >= 50:
        return "Sand"
    elif silt_percent >= 50:
        return "Silt"
    elif clay_percent >= 50:
        return "Clay"
    elif sand_percent >= 30:
        return "Sandy Loam"
    elif silt_percent >= 30:
        return "Silt Loam"
    elif clay_percent >= 30:
        return "Clay Loam"
    else:
        return "Loam"


soil_properties_df["soil texture"] = soil_properties_df.apply(soil_texture, axis=1)
print(soil_properties_df)


soil_properties_df.to_csv('soil_properties.csv', index = True)



In [ ]:
# download era5 dataset

variable = "wefesiteanalyst"
target_file = 'era5_wefesiteanalyst_'+name+'.nc'
print(target_file)


ds = era5.get_era5_data_from_datespan_and_position(
    variable=variable,
    start_date=start_date, end_date=end_date,
    latitude=lat, longitude=lon,
    target_file=target_file)

In [ ]:
# File transformation nc to csv
ds = xr.open_dataset('era5_wefesiteanalyst_'+name+'.nc')
era5_wefe = ds.to_dataframe()
# transform time: Costa Rica Time: UTC -6
# Create a date range with the desired starting date and frequency
print(era5_wefe)
date_range = pd.date_range(start='2022-04-30 18:00:00', freq='H', periods=len(era5_wefe))
# Set the date range as the index of the DataFrame
era5_wefe.set_index(date_range, inplace=True)
print(era5_wefe)
print(era5_wefe.dtypes)
# convert units
# global horizontal irradiance
era5_wefe['ghi'] = (era5_wefe['ssrd'] / 3600.0)
era5_wefe['t_air'] = era5_wefe['t2m']-273.15
era5_wefe['e'] *=1000
era5_wefe['tp']*=1000


def calc_sqrt_sum_squares(df, col1, col2):
    return np.sqrt(df[col1]**2 + df[col2]**2)

era5_wefe['windspeed'] = calc_sqrt_sum_squares(era5_wefe, 'u10', 'v10')

# create new dataframe only consisting out of data for required parameters

columns = ['ghi', 't_air', 'e', 'tp', 'windspeed']
values = era5_wefe[columns].values
ep = pd.DataFrame(values, columns=columns, index = era5_wefe.index)


print(ep)
#print(ep.dtypes)

ep.to_csv('era5_wefesiteanalyst_'+name+'.csv', index=True)


In [ ]:
#Increasing Image quality of figure to be generated
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300


In [ ]:
# Coopernicus donwload is outdated; we load a previously downloaded climate dataset (csv) for Arusí
era5_wefe = pd.read_csv(r"era5_wefesiteanalyst_Arusi.csv")
era5_wefe.head()

In [ ]:
ghi_data = era5_wefe['ghi']

# Calculate annual sum by summing all hourly values
annual_sum_ghi = ghi_data.sum()/1000

plt.figure(figsize=(8, 6))
plt.bar(ghi_data.index, ghi_data, color='orange')
plt.title('Hourly Solar Irradiance Profile (2022)', fontsize=18)
plt.xlabel('Hourly Timestamp [h]', fontsize=16)
plt.ylabel('Solar Irradiance [W/m²]', fontsize=16)
plt.grid(axis='y')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Add annual sum text
plt.text(
    0.99, 0.9,
    f'Annual Global Horizontal Irradiation:\n{annual_sum_ghi:.2f} kWh/m²',
    fontsize=12,
    ha='right',
    va='bottom',
    transform=plt.gca().transAxes,  # figure coordinate system for relative placement
    color='black'
)

plt.text(
    0.99, 0.01,
    f'Source: Era5',
    fontsize=12,
    ha='right',
    va='bottom',
    transform=plt.gca().transAxes,  # figure coordinate system for relative placement
    color='black'
)

plt.tight_layout()
plt.show()

In [ ]:
#Precipitation data from Era5
precipitation_data = era5_wefe['tp']

#Total Annual Precipitation
annual_sum_tp = precipitation_data.sum()

# Create a vertical bar chart
plt.figure(figsize=(8, 6))
plt.bar(precipitation_data.index, precipitation_data, color='steelblue')

plt.title('Hourly Precipitation Profile (2022)', fontsize=18)
plt.xlabel('Hourly Timestamp [h]', fontsize=16)
plt.ylabel('Precipitation [mm]', fontsize=16)
plt.grid(axis='y')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Add annual sum text
plt.text(
    0.99, 0.9,
    f'Total Annual Precipitation:\n{annual_sum_tp:.2f} mm',
    fontsize=12,
    ha='right',
    va='bottom',
    transform=plt.gca().transAxes,  # figure coordinate system for relative placement
    color='black'
)

plt.text(
    0.99, 0.01,
    f'Source: Era5',
    fontsize=12,
    ha='right',
    va='bottom',
    transform=plt.gca().transAxes,  # figure coordinate system for relative placement
    color='black'
)

plt.tight_layout()
plt.show()

In [ ]:
# plot temperature
temperature_data = era5_wefe['t_air']
# calculate average temperature
average_temp = temperature_data.mean()

plt.figure(figsize=(8, 6))

# Plot line diagram instead of bar chart
plt.plot(temperature_data.index, temperature_data, color='red', linestyle='-')

plt.title('Hourly Temperature Profile (2022)', fontsize=18)
plt.xlabel('Hourly Timestamp [h]', fontsize=16)
plt.ylabel('Temperature [°C]', fontsize=16)
plt.grid(axis='y')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Add text
plt.text(
    0.99, 0.9,
    f'Annual Average Temperature:\n{average_temp:.2f} °C',
    fontsize=12,
    ha='right',
    va='bottom',
    transform=plt.gca().transAxes,  # figure coordinate system for relative placement
    color='black'
)
plt.text(
    0.99, 0.01,
    f'Source: Era5',
    fontsize=12,
    ha='right',
    va='bottom',
    transform=plt.gca().transAxes,  # figure coordinate system for relative placement
    color='black'
)


plt.tight_layout()
plt.show()

In [ ]:
# plot windspeed
windspeed_data = era5_wefe['windspeed']

# calculate average wind speed
average_windspeed = windspeed_data.mean()

plt.figure(figsize=(8, 6))

# Plot line diagram instead of bar chart
plt.plot(windspeed_data.index, windspeed_data, color='darkgreen', linestyle='-')

plt.title('Hourly Windspeed Profile (2022)', fontsize=18)
plt.xlabel('Hourly Timestamp [h]', fontsize=16)
plt.ylabel('Windspeed [m/s]', fontsize=16)
plt.grid(axis='y')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Add text
plt.text(
    0.99, 0.9,
    f'Annual Average Windspeed:\n{average_windspeed:.2f} m/s',
    fontsize=12,
    ha='right',
    va='bottom',
    transform=plt.gca().transAxes,  # figure coordinate system for relative placement
    color='black'
)
plt.text(
    0.99, 0.01,
    f'Source: Era5',
    fontsize=12,
    ha='right',
    va='bottom',
    transform=plt.gca().transAxes,  # figure coordinate system for relative placement
    color='black'
)

plt.tight_layout()
plt.show()

In [ ]:
## Plot WEFE Demand Data for Arusí 
# raw data from surveys and interviews on site, stochastic demand modeling using WEFEDemand
# Load WEFE Demand Output Arusi
demand_wefe = pd.read_csv(r"WEFE_Demand_Arusi.csv")
demand_wefe.head()

In [ ]:
# plot service water demand
sw_demand_data = demand_wefe['service_water_demand']

# Calculate Total Drinking Water Demand
annual_sw_demand_sum =sw_demand_data.sum()

# Create a vertical bar chart
plt.figure(figsize=(8, 6))
plt.bar(sw_demand_data.index, sw_demand_data, color='steelblue')

plt.title('Hourly Service Water Demand Profile', fontsize=18)
plt.xlabel('Hourly Timestamp [h]', fontsize=16)
plt.ylabel('Service Water Demand [m³]', fontsize=16)
plt.grid(axis='y')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Add text
plt.text(
    0.99, 0.9,
    f'Annual Service Water Demand:\n{annual_sw_demand_sum:.2f} m³',
    fontsize=12,
    ha='right',
    va='bottom',
    transform=plt.gca().transAxes,  # figure coordinate system for relative placement
    color='black'
)

plt.tight_layout()
plt.show()

In [ ]:
# plot drinking water demand
dw_demand_data = demand_wefe['drinking_water_demand']

# Calculate Total Drinking Water Demand
annual_dw_demand_sum =dw_demand_data.sum()

# Create a vertical bar chart
plt.figure(figsize=(8, 6))
plt.bar(dw_demand_data.index, dw_demand_data, color='steelblue')

plt.title('Hourly Drinking Water Demand Profile', fontsize=18)
plt.xlabel('Hourly Timestamp [h]', fontsize=16)
plt.ylabel('Drinking Water Demand [m³]', fontsize=16)
plt.grid(axis='y')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Add text
plt.text(
    0.99, 0.9,
    f'Annual Drinking Water Demand:\n{annual_dw_demand_sum:.2f} m³',
    fontsize=12,
    ha='right',
    va='bottom',
    transform=plt.gca().transAxes,  # figure coordinate system for relative placement
    color='black'
)

plt.tight_layout()
plt.show()

In [ ]:
# Plot electricity demand profile
electricity_demand_data = demand_wefe['electricity_demand']

# Calculate Annual Electricity Demand
annual_electricity_demand_sum =electricity_demand_data.sum()

# Create a vertical bar chart
plt.figure(figsize=(8, 6))
plt.bar(electricity_demand_data.index, electricity_demand_data, color='#61007A')

plt.title('Hourly Electricity Demand Profile', fontsize=18)
plt.xlabel('Hourly Timestamp [h]', fontsize=16)
plt.ylabel('Electricity Demand [kWh]', fontsize=16)
plt.grid(axis='y')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Add text
plt.text(
    0.99, 0.9,
    f'Annual Electricity Demand:\n{annual_electricity_demand_sum:.2f} kWh',
    fontsize=12,
    ha='right',
    va='bottom',
    transform=plt.gca().transAxes,  # figure coordinate system for relative placement
    color='black'
)

plt.tight_layout()
plt.show()

In [ ]:
# Plot cooking energy demand profile
cooking_demand_data = demand_wefe['cooking_energy_demand']

# Calculate Annual Cooking Demand
annual_cooking_demand_sum =cooking_demand_data.sum()

# Create a vertical bar chart
plt.figure(figsize=(8, 6))
plt.bar(cooking_demand_data.index, cooking_demand_data, color='#E9920E')

plt.title('Hourly Cooking Energy Demand Profile', fontsize=18)
plt.xlabel('Hourly Timestamp [h]', fontsize=16)
plt.ylabel('Cooking Energy Demand [kWh]', fontsize=16)
plt.grid(axis='y')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)

# Add text
plt.text(
    0.99, 0.9,
    f'Annual Cooking Energy Demand:\n{annual_cooking_demand_sum:.2f} kWh',
    fontsize=12,
    ha='right',
    va='bottom',
    transform=plt.gca().transAxes,  # figure coordinate system for relative placement
    color='black'
)

plt.tight_layout()
plt.show()